In [14]:
# ============================================================
#                      Spaceship Titanic 
# ============================================================

import numpy as np
import pandas as pd
import os
import warnings
warnings.filterwarnings('ignore')
os.environ['PYTHONWARNINGS'] = 'ignore'

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.model_selection import (
    StratifiedKFold, GroupKFold, cross_val_score
)
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import ExtraTreesClassifier, VotingClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

SEED = 42
np.random.seed(SEED)
print("Setup complete. Seed = 42")

Setup complete. Seed = 42


In [15]:
# --- Load Data ---
DATA = '/kaggle/input/competitions/spaceship-titanic'

train = pd.read_csv(f'{DATA}/train.csv')
test  = pd.read_csv(f'{DATA}/test.csv')

print("Train:", train.shape)
print("Test :", test.shape)

Train: (8693, 14)
Test : (4277, 13)


In [16]:
# --- Feature Engineering ---
SPEND = ['RoomService', 'FoodCourt', 'ShoppingMall', 'Spa', 'VRDeck']

def build(df):
    d = df.copy()

    # Passenger ID parts
    d['GID']  = d['PassengerId'].str.split('_').str[0].astype(int)
    d['Seat'] = d['PassengerId'].str.split('_').str[1].astype(int)

    # Group size
    gsize = d.groupby('GID')['PassengerId'].transform('count')
    d['GroupLen'] = gsize
    d['Solo']     = (d['GroupLen'] == 1).astype(int)

    # Cabin parts
    parts = d['Cabin'].str.split('/', expand=True)
    d['Zone'] = parts[0]
    d['Room'] = pd.to_numeric(parts[1], errors='coerce')
    d['Pier'] = parts[2]

    # Spending features
    sp = d[SPEND].fillna(0)
    d['Cash']      = sp.sum(axis=1)
    d['CashLog']   = np.log1p(d['Cash'])
    d['ZeroSpend'] = (d['Cash'] == 0).astype(int)
    d['UsedCount'] = (sp > 0).sum(axis=1)

    # Luxury vs basic
    d['HighEnd']   = d[['Spa', 'VRDeck', 'RoomService']].fillna(0).sum(axis=1)
    d['LowEnd']    = d[['FoodCourt', 'ShoppingMall']].fillna(0).sum(axis=1)
    d['HighRatio'] = d['HighEnd'] / (d['Cash'] + 1)

    # Interaction
    d['SleepZero'] = d['CryoSleep'].map({True: 1, False: 0}).fillna(0) * d['ZeroSpend']

    return d

train = build(train)
test  = build(test)
print("Columns after FE:", train.shape[1])

Columns after FE: 29


In [17]:
# --- Prepare Features ---
NUMF = [
    'Age', 'GroupLen', 'Solo', 'Seat', 'Room',
    'Cash', 'CashLog', 'ZeroSpend', 'UsedCount',
    'HighEnd', 'LowEnd', 'HighRatio', 'SleepZero'
] + SPEND

CATF = ['HomePlanet', 'CryoSleep', 'Destination', 'VIP', 'Zone', 'Pier']

X = train[NUMF + CATF]
y = train['Transported'].astype(int)

print("X shape:", X.shape)
print("Balance:", y.mean())

X shape: (8693, 24)
Balance: 0.5036236051995858


In [18]:
# --- Pipeline Builder ---
def make_pipe(model):
    num_step = Pipeline([
        ('fill', SimpleImputer(strategy='mean')),
        ('norm', StandardScaler())
    ])
    cat_step = Pipeline([
        ('fill', SimpleImputer(strategy='most_frequent')),
        ('code', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
    ])
    prep = ColumnTransformer([
        ('n', num_step, NUMF),
        ('c', cat_step, CATF)
    ])
    return Pipeline([('prep', prep), ('algo', model)])


def score_model(model, label):
    p = make_pipe(model)
    cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
    s = cross_val_score(p, X, y, cv=cv, scoring='accuracy', n_jobs=-1)
    print(f"{label:42} {s.mean():.4f}  (+/- {s.std():.4f})")
    return s.mean()


print("Pipeline ready.")

Pipeline ready.


In [19]:
# --- Iteration Log Setup ---
log = []

def record(num, model, change, cv, lb='—'):
    log.append({
        'Iteration': num,
        'Model': model,
        'Change': change,
        'CV Score': round(cv, 4),
        'LB': lb
    })
    print(f"Recorded {num}: {model} -> CV {cv:.4f}")

print("Log ready.")

Log ready.


In [20]:
# --- Five Base Models ---
print("=" * 60)
print("FIVE BASE MODELS")
print("=" * 60)

m1 = LogisticRegression(C=1.0, max_iter=1500, random_state=SEED)
s1 = score_model(m1, "1. Logistic Regression")
record(1, "Logistic Regression", "Baseline, default params", s1)

m2 = DecisionTreeClassifier(max_depth=8, min_samples_leaf=8, random_state=SEED)
s2 = score_model(m2, "2. Decision Tree")
record(2, "Decision Tree", "max_depth=8, min_samples_leaf=8", s2)

m3 = ExtraTreesClassifier(n_estimators=400, max_depth=12,
                          min_samples_leaf=3, random_state=SEED, n_jobs=-1)
s3 = score_model(m3, "3. Extra Trees")
record(3, "Extra Trees", "n_estimators=400, max_depth=12", s3)

m4 = CatBoostClassifier(iterations=400, learning_rate=0.06, depth=5,
                        verbose=0, random_seed=SEED)
s4 = score_model(m4, "4. CatBoost")
record(4, "CatBoost", "iterations=400, lr=0.06, depth=5", s4, 0.80430)

m5 = LGBMClassifier(n_estimators=400, learning_rate=0.06, num_leaves=31,
                    max_depth=-1, random_state=SEED, verbose=-1)
s5 = score_model(m5, "5. LightGBM")
record(5, "LightGBM", "n_estimators=400, lr=0.06, leaves=31", s5)

FIVE BASE MODELS
1. Logistic Regression                     0.7959  (+/- 0.0049)
Recorded 1: Logistic Regression -> CV 0.7959
2. Decision Tree                           0.7977  (+/- 0.0081)
Recorded 2: Decision Tree -> CV 0.7977
3. Extra Trees                             0.8090  (+/- 0.0110)
Recorded 3: Extra Trees -> CV 0.8090
4. CatBoost                                0.8133  (+/- 0.0037)
Recorded 4: CatBoost -> CV 0.8133
5. LightGBM                                0.8098  (+/- 0.0022)
Recorded 5: LightGBM -> CV 0.8098


In [21]:
# --- Best Model: 4-Model Voting [1, 2, 4, 3] ---
print("=" * 60)
print("4-MODEL VOTING [1, 2, 4, 3]")
print("=" * 60)

voter_best = VotingClassifier(
    estimators=[
        ('dt',   DecisionTreeClassifier(max_depth=8, min_samples_leaf=8,
                                        random_state=SEED)),
        ('et',   ExtraTreesClassifier(n_estimators=400, max_depth=12,
                                      min_samples_leaf=3, random_state=SEED, n_jobs=-1)),
        ('cat',  CatBoostClassifier(iterations=400, learning_rate=0.06, depth=5,
                                    verbose=0, random_seed=SEED)),
        ('lgbm', LGBMClassifier(n_estimators=400, learning_rate=0.06, num_leaves=31,
                                max_depth=-1, random_state=SEED, verbose=-1)),
    ],
    voting='soft',
    weights=[1, 2, 4, 3]
)

s_voter = score_model(voter_best, "4-Model Voting [1,2,4,3]")
record(6, "4-Model Voting", "DT + ET + CatBoost + LGBM, weights [1,2,4,3]",
       s_voter, 0.81014)

# --- GroupKFold check ---
groups = train['GID'].values
cv_group = GroupKFold(n_splits=5)
pipe = make_pipe(voter_best)
s_group = cross_val_score(pipe, X, y, cv=cv_group, groups=groups,
                           scoring='accuracy', n_jobs=-1).mean()
print(f"GroupKFold CV: {s_group:.4f}")
print(f"Gap          : {s_voter - s_group:+.4f}")

4-MODEL VOTING [1, 2, 4, 3]
4-Model Voting [1,2,4,3]                   0.8164  (+/- 0.0064)
Recorded 6: 4-Model Voting -> CV 0.8164
GroupKFold CV: 0.8135
Gap          : +0.0029


In [22]:
# --- Final Comparison ---
print("=" * 60)
print("FINAL COMPARISON")
print("=" * 60)

results = {
    'Logistic Regression': s1,
    'Decision Tree':       s2,
    'Extra Trees':         s3,
    'CatBoost':            s4,
    'LightGBM':            s5,
    '4-Model Voting':      s_voter,
}

for name, s in sorted(results.items(), key=lambda kv: kv[1], reverse=True):
    print(f"{name:25}: {s:.4f}")

best_name = max(results, key=results.get)
print("\nBest model:", best_name)

FINAL COMPARISON
4-Model Voting           : 0.8164
CatBoost                 : 0.8133
LightGBM                 : 0.8098
Extra Trees              : 0.8090
Decision Tree            : 0.7977
Logistic Regression      : 0.7959

Best model: 4-Model Voting


In [23]:
# --- Final Iteration Log ---
print("=" * 60)
print("FINAL ITERATION LOG")
print("=" * 60)

log_df = pd.DataFrame(log)
print(log_df.to_string(index=False))

best_row = max(log, key=lambda r: r['CV Score'])
print("\n" + "=" * 60)
print(f"Best: {best_row['Model']} -> CV {best_row['CV Score']:.4f}")
print("=" * 60)

FINAL ITERATION LOG
 Iteration               Model                                       Change  CV Score       LB
         1 Logistic Regression                     Baseline, default params    0.7959        —
         2       Decision Tree              max_depth=8, min_samples_leaf=8    0.7977        —
         3         Extra Trees               n_estimators=400, max_depth=12    0.8090        —
         4            CatBoost             iterations=400, lr=0.06, depth=5    0.8133   0.8043
         5            LightGBM         n_estimators=400, lr=0.06, leaves=31    0.8098        —
         6      4-Model Voting DT + ET + CatBoost + LGBM, weights [1,2,4,3]    0.8164  0.81014

Best: 4-Model Voting -> CV 0.8164


In [24]:
# --- Create Submission ---
print("=" * 60)
print("CREATING SUBMISSION")
print("=" * 60)

final_model = make_pipe(voter_best)
final_model.fit(X, y)

X_test = test[NUMF + CATF]
predictions = final_model.predict(X_test)

submission = pd.DataFrame({
    'PassengerId': test['PassengerId'],
    'Transported': predictions.astype(bool)
})
submission.to_csv('submission.csv', index=False)

print("submission.csv created.")
print(f"Predicted Transported    : {submission['Transported'].sum()}")
print(f"Predicted Not Transported: {(~submission['Transported']).sum()}")
print(f"Ratio                    : {submission['Transported'].mean():.3f}")
print("\nFirst 10 rows:")
print(submission.head(10))

CREATING SUBMISSION
submission.csv created.
Predicted Transported    : 2222
Predicted Not Transported: 2055
Ratio                    : 0.520

First 10 rows:
  PassengerId  Transported
0     0013_01         True
1     0018_01        False
2     0019_01         True
3     0021_01         True
4     0023_01         True
5     0027_01         True
6     0029_01         True
7     0032_01         True
8     0032_02         True
9     0033_01         True


In [25]:
# --- Validate Submission ---
import os

print("=" * 60)
print("SUBMISSION VALIDATION")
print("=" * 60)

print(f"File exists: {os.path.exists('submission.csv')}")

sub = pd.read_csv('submission.csv')
print(f"Shape      : {sub.shape}")
print(f"Columns    : {list(sub.columns)}")
print(f"Transported: {sub['Transported'].dtype}")
print(f"True       : {sub['Transported'].sum()}")
print(f"False      : {(~sub['Transported']).sum()}")
print(f"Missing    : {sub.isna().sum().sum()}")
print(f"Duplicates : {sub['PassengerId'].duplicated().sum()}")

if (sub.shape == (4277, 2) and
    list(sub.columns) == ['PassengerId', 'Transported'] and
    sub.isna().sum().sum() == 0 and
    sub['PassengerId'].duplicated().sum() == 0):
    print("\nAll checks passed. Ready to submit.")
else:
    print("\nProblems found. Fix before submitting.")

SUBMISSION VALIDATION
File exists: True
Shape      : (4277, 2)
Columns    : ['PassengerId', 'Transported']
Transported: bool
True       : 2222
False      : 2055
Missing    : 0
Duplicates : 0

All checks passed. Ready to submit.
